In [ ]:
import pandas as pd

def clean_pay_data(file_path, pay_column_name):
    
    # Read data
    df = pd.read_csv(file_path)

    # Keep only actual pay values, not confidence values
    df = df[df['MEASURES'] == 20100].copy()

    # Keep only region and pay
    df = df[['GEOGRAPHY_NAME', 'OBS_VALUE']].copy()

    # Rename columns
    df.columns = ['Region', pay_column_name]

    # Make sure pay is numeric
    df[pay_column_name] = pd.to_numeric(
        df[pay_column_name],
        errors='coerce'
    )

    # Convert weekly pay to yearly pay
    df[pay_column_name] = df[pay_column_name] * 52

    # Sort by region
    df = df.sort_values('Region').reset_index(drop=True)

    return df


# Clean workplace pay data
df_workplace = clean_pay_data(
    "data/nomis_ashe_workplace.csv",
    "Pay_Workplace"
)

# Clean resident pay data
df_resident = clean_pay_data(
    "data/nomis_ashe_resident.csv",
    "Pay_Resident"
)


# Merge the two datasets
df_combined = pd.merge(
    df_resident,
    df_workplace,
    on='Region',
    how='inner'
)


# Calculate difference between workplace and resident pay
df_combined['Pay_Difference'] = (
    df_combined['Pay_Workplace'] -
    df_combined['Pay_Resident']
)


# Calculate percentage difference
df_combined['Pay_Difference_Percent'] = (
    df_combined['Pay_Difference'] /
    df_combined['Pay_Resident']
) * 100


# Sort by region
df_combined = df_combined.sort_values('Region').reset_index(drop=True)


# Display
print(df_combined)

                      Region  Pay_Resident  Pay_Workplace  Pay_Difference  \
0                       East       41854.8        39863.2         -1991.6   
1              East Midlands       37481.6        36597.6          -884.0   
2                     London       46940.4        49826.4          2886.0   
3                 North East       35989.2        35422.4          -566.8   
4                 North West       38209.6        38178.4           -31.2   
5           Northern Ireland       37211.2        37081.2          -130.0   
6                   Scotland       40331.2        40237.6           -93.6   
7                 South East       42390.4        40341.6         -2048.8   
8                 South West       38168.0        37892.4          -275.6   
9                      Wales       37403.6        36623.6          -780.0   
10             West Midlands       38012.0        37980.8           -31.2   
11  Yorkshire and The Humber       37003.2        36826.4          -176.8   

In [20]:
df = pd.read_csv("data/nomis_ashe_workplace.csv")

# Keep only actual pay values, not confidence values
df_pay = df[df['MEASURES'] == 20100].copy()

# Keep only the columns we need
df_clean = df_pay[['GEOGRAPHY_NAME', 'OBS_VALUE']].copy()

# Rename them
df_clean.columns = ['Region', 'Weekly_Pay']

# Make sure pay is numeric
df_clean['Weekly_Pay'] = pd.to_numeric(df_clean['Weekly_Pay'], errors='coerce')

# Convert weekly pay to annual pay
df_clean['Pay_Yearly'] = df_clean['Weekly_Pay'] * 52

# Remove weekly pay
df_clean = df_clean.drop(columns='Weekly_Pay')

# Put columns in desired order
df_clean = df_clean[['Region', 'Pay_Yearly']]

# Sort
df_clean = df_clean.sort_values('Region').reset_index(drop=True)

# Save
df_clean.to_csv('cleaned_pay_data_working_there.csv', index=False)

print(df_clean)

print(f"Unique regions: {df_clean['Region'].nunique()}")

                      Region  Pay_Yearly
0                       East     39863.2
1              East Midlands     36597.6
2                     London     49826.4
3                 North East     35422.4
4                 North West     38178.4
5           Northern Ireland     37081.2
6                   Scotland     40237.6
7                 South East     40341.6
8                 South West     37892.4
9                      Wales     36623.6
10             West Midlands     37980.8
11  Yorkshire and The Humber     36826.4
Unique regions: 12
